# scRNA Demultiplexing
Vireo-based cell-line demultiplexing results, donor assignment quality, concordance profiles,
and reference-guided validation using Pool_ctr.

In [ ]:
import os
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
print('Working directory:', os.getcwd())

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output

DB_PATH = 'results/variants.db'

def query(sql, params=()):
    with sqlite3.connect(DB_PATH) as conn:
        return pd.read_sql_query(sql, conn, params=params)

# Load demux runs — table only exists after scrna pipeline has been run
try:
    demux_runs = query("SELECT DISTINCT demux_run_id FROM cell_assignments")
except Exception:
    demux_runs = pd.DataFrame(columns=['demux_run_id'])

print(f'Demux runs found: {len(demux_runs)}')
if demux_runs.empty:
    print('No cell assignment data in DB yet — run the scrna pipeline first.')

## scRNA demultiplexing results
Cell composition, doublet rate, Vireo donor probabilities, and concordance with DB runs.

In [ ]:
if demux_runs.empty:
    print('No demux runs available.')
else:
    pool_select = widgets.Dropdown(
        options=demux_runs['demux_run_id'].tolist(),
        description='Pool:',
        layout=widgets.Layout(width='55%')
    )
    out_pool = widgets.Output()

    def show_pool_demux(demux_run_id):
        cells = query("""
            SELECT ca.cell_barcode, ca.assigned_donor, ca.prob_max,
                   ca.prob_doublet, ca.n_vars, ca.match_run_id, ca.concordance
            FROM cell_assignments ca
            WHERE ca.demux_run_id = ?
        """, (demux_run_id,))
        if cells.empty:
            print('No cell assignments for this pool.')
            return

        donor_counts = cells['assigned_donor'].value_counts()
        doublets = cells[cells['assigned_donor'] == 'doublet']
        assigned = cells[cells['assigned_donor'].isin(['doublet', 'unassigned']) == False]

        fig = make_subplots(
            rows=2, cols=2,
            subplot_titles=(
                'Cell composition per donor',
                'Doublet probability distribution',
                'Max donor probability (assigned cells)',
                'Concordance with DB run'
            ),
            horizontal_spacing=0.12, vertical_spacing=0.15
        )
        # Panel 1: stacked bar of donor cell counts
        fig.add_trace(go.Bar(
            x=donor_counts.index, y=donor_counts.values,
            marker_color=px.colors.qualitative.Safe[:len(donor_counts)],
            name='Cell count'
        ), row=1, col=1)

        # Panel 2: doublet probability histogram
        fig.add_trace(go.Histogram(
            x=cells['prob_doublet'], nbinsx=50,
            marker_color='#e63946', name='Prob doublet'
        ), row=1, col=2)

        # Panel 3: max probability for assigned cells
        if not assigned.empty:
            for donor in assigned['assigned_donor'].unique():
                sub = assigned[assigned['assigned_donor'] == donor]
                fig.add_trace(go.Histogram(
                    x=sub['prob_max'], nbinsx=30, name=donor,
                    opacity=0.7
                ), row=2, col=1)
        fig.update_layout(barmode='overlay')

        # Panel 4: concordance bar if available
        if 'concordance' in cells.columns and cells['concordance'].notna().any():
            conc_by_donor = (
                cells[cells['concordance'].notna()]
                .groupby('assigned_donor')['concordance']
                .mean()
                .sort_values(ascending=False)
            )
            fig.add_trace(go.Bar(
                x=conc_by_donor.index, y=conc_by_donor.values,
                marker_color='#2d6a4f', name='Concordance'
            ), row=2, col=2)

        fig.update_layout(
            height=600,
            title=f'Pool: {demux_run_id}  —  {len(cells):,} cells  '  
                  f'({len(doublets)} doublets, {(len(doublets)/len(cells)*100):.1f}%)',
        )
        fig.update_xaxes(tickangle=45, row=1, col=1)
        fig.show()
        print(f'Cell assignment summary:')
        display(cells.groupby('assigned_donor').agg(
            n_cells=('cell_barcode', 'count'),
            mean_prob=('prob_max', 'mean'),
            mean_concordance=('concordance', 'mean')
        ).sort_values('n_cells', ascending=False)
        .style.format({'mean_prob': '{:.4f}', 'mean_concordance': '{:.4f}'}))

    def on_pool_change(change):
        with out_pool:
            clear_output(wait=True)
            show_pool_demux(change['new'])

    pool_select.observe(on_pool_change, names='value')
    display(pool_select, out_pool)
    show_pool_demux(pool_select.value)

## Concordance profiles — all pools
Distribution of per-cell concordance scores. A clean separation between known same-donor
and different-donor pairs validates the genotype-based assignment.

In [ ]:
def compute_concordance(demux_run_id):
    """
    Returns a DataFrame with (assigned_donor, vireo_donor, concordance, same_donor) per cell.
    same_donor = True when the Vireo assignment matches the DB run label by cell_line.
    """
    cells = query("""
        SELECT ca.cell_barcode, ca.assigned_donor, ca.concordance, ca.match_run_id,
               s.cell_line
        FROM cell_assignments ca
        LEFT JOIN runs r ON ca.match_run_id = r.run_id
        LEFT JOIN samples s ON r.sample_id = s.sample_id
        WHERE ca.demux_run_id = ? AND ca.concordance IS NOT NULL
          AND ca.assigned_donor NOT IN ('doublet', 'unassigned')
    """, (demux_run_id,))
    return cells

if not demux_runs.empty:
    fig = go.Figure()
    for pool_id in demux_runs['demux_run_id']:
        df = compute_concordance(pool_id)
        if df.empty:
            continue
        fig.add_trace(go.Violin(
            y=df['concordance'], name=pool_id,
            box_visible=True, meanline_visible=True,
            points='outliers'
        ))
    fig.update_layout(
        height=450, title='Concordance distribution per pool',
        yaxis_title='Concordance (Vireo assignment vs DB run)',
        xaxis_tickangle=30
    )
    fig.show()

## Concordance matrix — Plot A: raw per-cell concordance
Heatmap of mean concordance for each Vireo donor × DB run combination across all pools.

In [ ]:
if not demux_runs.empty:
    all_concordance = []
    for pool_id in demux_runs['demux_run_id']:
        df = compute_concordance(pool_id)
        if not df.empty:
            df['demux_run_id'] = pool_id
            all_concordance.append(df)

    if all_concordance:
        all_df = pd.concat(all_concordance, ignore_index=True)

        pivot = (
            all_df.groupby(['assigned_donor', 'cell_line'])['concordance']
            .mean()
            .unstack(fill_value=np.nan)
        )
        fig = go.Figure(go.Heatmap(
            z=pivot.values,
            x=pivot.columns.tolist(),
            y=pivot.index.tolist(),
            colorscale='RdYlGn', zmin=0, zmax=1,
            text=np.round(pivot.values, 3),
            texttemplate='%{text}'
        ))
        fig.update_layout(
            height=max(350, 30 * len(pivot)),
            title='Plot A: Mean concordance — Vireo donor (rows) × DB run label (cols)',
            xaxis_tickangle=45
        )
        fig.show()
    else:
        print('No concordance data found across any pool.')

## Validation — reference-guided Vireo (Pool_ctr)
For the control pool where expected donor identities are known, compares Vireo-assigned
donors against reference to verify assignment quality.

In [ ]:
pool_ctr_ids = [p for p in demux_runs['demux_run_id'].tolist() if 'ctr' in p.lower() or 'ctrl' in p.lower()]
print(f'Control pools found: {pool_ctr_ids}')

if pool_ctr_ids:
    for pool_id in pool_ctr_ids:
        cells = query("""
            SELECT ca.assigned_donor, ca.prob_max, ca.prob_doublet,
                   ca.concordance, s.cell_line
            FROM cell_assignments ca
            LEFT JOIN runs r ON ca.match_run_id = r.run_id
            LEFT JOIN samples s ON r.sample_id = s.sample_id
            WHERE ca.demux_run_id = ?
        """, (pool_id,))
        if cells.empty:
            print(f'{pool_id}: no assignments.')
            continue

        assigned = cells[~cells['assigned_donor'].isin(['doublet', 'unassigned'])]
        if assigned.empty:
            print(f'{pool_id}: no non-doublet assignments.')
            continue

        # Per-donor concordance vs expected label
        summary = (
            assigned.groupby(['assigned_donor', 'cell_line'])
            .agg(n=('prob_max', 'count'), mean_conc=('concordance', 'mean'), mean_prob=('prob_max', 'mean'))
            .reset_index()
            .sort_values('n', ascending=False)
        )
        print(f'\n--- {pool_id} ---')
        display(summary.style.format({'mean_conc': '{:.4f}', 'mean_prob': '{:.4f}'}))

        fig = go.Figure(go.Bar(
            x=summary['assigned_donor'] + ' → ' + summary['cell_line'].fillna('unknown'),
            y=summary['mean_conc'],
            marker_color='#457b9d'
        ))
        fig.update_layout(
            height=350,
            title=f'Reference validation — {pool_id}: mean concordance per Vireo donor assignment',
            xaxis_tickangle=45,
            yaxis_title='Mean concordance'
        )
        fig.show()
else:
    print('No control pool found (looking for \'ctr\' or \'ctrl\' in demux_run_id).')

## Final assignments summary
Reads `results/demux/final_assignments.tsv` and summarises the per-cell donor calls.

In [ ]:
fa_path = 'results/demux/final_assignments.tsv'
if os.path.exists(fa_path):
    fa = pd.read_csv(fa_path, sep='\t')
    print(f'Loaded final_assignments.tsv: {fa.shape[0]:,} rows, {fa.shape[1]} columns')
    display(fa.head())

    if 'final_donor' in fa.columns:
        summary = fa['final_donor'].value_counts().reset_index()
        summary.columns = ['final_donor', 'n_cells']
        fig = px.bar(summary, x='final_donor', y='n_cells', title='Final donor assignments across all pools')
        fig.update_xaxes(tickangle=45)
        fig.show()
else:
    print(f'File not found: {fa_path}')
    print('Run the scrna pipeline to generate final_assignments.tsv.')